# Demo OmniVoice – Zero-shot TTS tiếng Việt

**Paper:** OmniVoice: Towards Omnilingual Zero-Shot Text-to-Speech with Diffusion Language Models (Zhu et al., 2026, arXiv:2604.00688)

**Nội dung demo:**
1. Clone giọng tiếng Việt từ 3–10 giây audio mẫu
2. So sánh số bước inference (8 / 16 / 32) – tốc độ vs chất lượng
3. Ảnh hưởng của text normalization (số, %, viết tắt)
4. Đánh giá khách quan bằng Whisper (WER)
5. Giao diện Gradio để thầy tự gõ câu thử

> **Kaggle:** Settings → Accelerator: **GPU T4 x2** (hoặc P100), và bật **Internet: On** để cài thư viện, tải model.

> **License:** code Apache 2.0, model CC-BY-NC (chỉ dùng cho nghiên cứu phi thương mại). Chỉ clone giọng khi có sự đồng ý của người đó.

## 0. Kiểm tra GPU và cài đặt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q omnivoice gradio jiwer

## 1. Load model

In [ ]:
import time, re
import numpy as np
import torch
from IPython.display import Audio, display
from omnivoice import OmniVoice

SR = 24000  # OmniVoice xuất audio 24 kHz

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
)

def to_np(a):
    """Chuyển output (torch.Tensor hoặc np.ndarray) về mảng 1 chiều."""
    if hasattr(a, "detach"):
        a = a.detach().float().cpu().numpy()
    return np.asarray(a, dtype=np.float32).squeeze()

def synth(text, ref_audio=None, ref_text=None, num_step=32, speed=1.0, **kw):
    """Sinh audio, trả về (waveform, thời gian sinh, RTF)."""
    args = dict(text=text, num_step=num_step, speed=speed, **kw)
    if ref_audio:
        args["ref_audio"] = ref_audio
        if ref_text:
            args["ref_text"] = ref_text  # bỏ trống thì OmniVoice tự dùng Whisper để chép lời
    torch.cuda.synchronize()
    t0 = time.time()
    wav = to_np(model.generate(**args)[0])
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    rtf = elapsed / (len(wav) / SR)
    return wav, elapsed, rtf

## 2. Audio mẫu (giọng tiếng Việt)

Điền path file mẫu vào `SRC_AUDIO` (`.m4a`, `.wav`, `.mp3`, ...). File upload lên Kaggle nằm ở `/kaggle/input/<tên-dataset>/`. Cell dùng ffmpeg tách tiếng ra WAV mono 24 kHz.

`REF_TEXT` là lời trong file mẫu; để trống thì model tự chép lời bằng Whisper.

In [ ]:
import os, subprocess

SRC_AUDIO = "/kaggle/input/datasets/quachthanhhmd/test-voice/test-voice.m4a"   # đổi thành path file của bạn
REF_TEXT = ""

assert os.path.exists(SRC_AUDIO), f"Không tìm thấy file: {SRC_AUDIO}"

REF_AUDIO = "/kaggle/working/ref.wav"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", SRC_AUDIO,
                "-vn", "-ac", "1", "-ar", "24000", REF_AUDIO], check=True)
display(Audio(REF_AUDIO))

## 3. Clone giọng tiếng Việt

In [ ]:
TEXT = "Sway Sampling là chiến lược lấy mẫu flow step mà F5-TTS (Chen et al., 2025) đề xuất cho giai đoạn inference. Trong quá trình huấn luyện, mô hình vẫn lấy flow step đều trên toàn bộ quỹ đạo từ noise đến dữ liệu. Khi suy luận, Sway Sampling dùng một hàm biến đổi đơn điệu để phân bố lại các bước, và một hệ số điều chỉnh quyết định các bước dồn về phía nào. Với hệ số âm, các bước tập trung ở vùng gần noise. Với hệ số dương, chúng tập trung ở vùng gần dữ liệu. Khi hệ số bằng không, ta có lại cách lấy mẫu đều. Phương pháp dựa trên nhận định rằng các bước đầu của ODE solver quyết định khung tổng thể và alignment giữa text và speech, còn các bước sau chủ yếu tinh chỉnh chi tiết âm học. Vì vậy, dành nhiều lượt function evaluation hơn cho giai đoạn đầu giúp quỹ đạo sinh bám sát nội dung văn bản hơn. Thực nghiệm cho thấy khi bật Sway Sampling, WER của F5-TTS trên LibriSpeech-PC giảm từ 2.84 xuống 2.41, còn SIM-o tăng từ 0.62 lên 0.66. Ưu điểm nổi bật của phương pháp là không cần huấn luyện lại, nên có thể áp dụng trực tiếp cho các mô hình flow matching khác như E2 TTS. Tuy nhiên, hiệu quả của nó chủ yếu được chứng minh bằng thực nghiệm, chưa có cơ sở lý thuyết chặt chẽ, và hệ số điều chỉnh vẫn phải chọn thủ công."

wav, elapsed, rtf = synth(TEXT, REF_AUDIO, REF_TEXT, num_step=32)
print(f"Thời gian sinh: {elapsed:.2f}s | Độ dài audio: {len(wav)/SR:.2f}s | RTF: {rtf:.3f}")
display(Audio(wav, rate=SR))

## 4. Thí nghiệm: số bước inference (`num_step`)

OmniVoice sinh bằng cách **mở mask dần** các acoustic token qua N bước. Paper báo cáo chất lượng giảm rõ khi xuống 8 bước.

In [ ]:
results = {}
for steps in [8, 16, 32]:
    wav_s, t_s, rtf_s = synth(TEXT, REF_AUDIO, REF_TEXT, num_step=steps)
    results[steps] = wav_s
    print(f"num_step={steps:>2} | thời gian {t_s:.2f}s | RTF {rtf_s:.3f}")
    display(Audio(wav_s, rate=SR))

## 5. Thí nghiệm: text normalization

Chính tác giả thừa nhận OmniVoice yếu với chuỗi số. Tuỳ chọn `normalize_text=True` của thư viện dùng WeTextProcessing (hướng tới tiếng Trung/Anh), nên với tiếng Việt ta **tự viết số thành chữ**.

In [ ]:
RAW  = "Nhiệt độ hôm nay là 30°C, độ ẩm 80%, giá vé 15k tại TP.HCM."
NORM = "Nhiệt độ hôm nay là ba mươi độ xê, độ ẩm tám mươi phần trăm, giá vé mười lăm nghìn tại thành phố Hồ Chí Minh."

for label, t in [("Chưa normalize", RAW), ("Đã normalize", NORM)]:
    w, _, _ = synth(t, REF_AUDIO, REF_TEXT, num_step=32)
    print(label, "→", t)
    display(Audio(w, rate=SR))

## 6. Đánh giá khách quan: WER bằng Whisper

Cho Whisper nghe lại audio sinh ra rồi so với text gốc. WER càng thấp, model đọc càng đúng (paper cũng đo theo cách này).

In [ ]:
import jiwer
from transformers import pipeline

asr = pipeline("automatic-speech-recognition",
               model="openai/whisper-large-v3-turbo",
               dtype=torch.float16, device=0)

def clean(s):
    return re.sub(r"[^\w\s]", " ", s.lower()).split()

def wer_of(wav, ref_text):
    hyp = asr({"raw": wav, "sampling_rate": SR},
              return_timestamps=True,
              generate_kwargs={"language": "vietnamese"})["text"]
    return jiwer.wer(" ".join(clean(ref_text)), " ".join(clean(hyp))), hyp

for steps, w in results.items():
    score, hyp = wer_of(w, TEXT)
    print(f"num_step={steps:>2} | WER={score*100:.1f}% | Whisper nghe: {hyp}")

## 7. Giao diện Gradio cho thầy tự thử

Chạy cell dưới, mở link `*.gradio.live` để dùng trên trình duyệt.

In [ ]:
import gradio as gr

def ui_tts(ref_audio, ref_text, text, steps, speed):
    if not text.strip():
        raise gr.Error("Nhập câu cần đọc")
    wav, elapsed, rtf = synth(text, ref_audio, ref_text.strip() or None,
                              num_step=int(steps), speed=float(speed))
    return (SR, wav), f"Thời gian: {elapsed:.2f}s | RTF: {rtf:.3f}"

demo = gr.Interface(
    fn=ui_tts,
    inputs=[
        gr.Audio(type="filepath", value=REF_AUDIO, label="Audio mẫu (mặc định: REF_AUDIO)"),
        gr.Textbox(value=REF_TEXT, label="Lời của audio mẫu (để trống = tự chép lời)"),
        gr.Textbox(label="Câu cần đọc", lines=3,
                   value="Xin chào, đây là giọng nói được tạo bởi OmniVoice."),
        gr.Slider(8, 64, value=32, step=8, label="num_step"),
        gr.Slider(0.7, 1.3, value=1.0, step=0.05, label="speed"),
    ],
    outputs=[gr.Audio(label="Kết quả"), gr.Textbox(label="Thông tin")],
    title="OmniVoice – Demo TTS tiếng Việt",
)
demo.launch(share=True, debug=False)